> **日本語版について**: このノートブックは、[jpmorganchase/python-training](https://github.com/jpmorganchase/python-training) の英語原文を日本語に翻訳したものです（Apache License 2.0）。コードの動作は原文から変更していません。

# 例1：ストラドルの価格計算

少し複雑な計算を使います。それでも式自体は簡単です。ATMF ストラドルの価格を近似で求めます。

$$ STRADDLE_{ATMF} \approx \frac{2}{\sqrt{2\pi}} F \times \sigma \sqrt(T) $$
$$ \sigma = インプライド・ボラティリティ $$
$$ T = 満期までの期間 $$
$$ F = 原資産のフォワード $$

まず、インプライド・ボラティリティと満期までの期間を決めます。ここでは F を 1 とします。価格は F に比例するので、あとから調整できます。

In [ ]:
vol = 0.2
time = 1.

In [ ]:
2. * ( (1 / ( 2 * 3.14 ) ** 0.5 ) * vol * time ** 0.5 )

ストラドルを計算するたびに、この式を打ち直すのは面倒です。間違いも起きやすくなります。関数にしておけば、何度でも使い回せます。

In [ ]:
def straddlePricer( vol, time ):
    return 2. * ( ( 1. / ( 2 * 3.14 ) ** 0.5 ) * vol * time ** 0.5 )

実行しても、出力には何も表示されません。それで問題ありません。関数は定義されています。次のセルで、関数の結果と上のセルの結果を比べてみます。

In [ ]:
print( straddlePricer( 0.2, 1.0 ) )
print( 2. * ( ( 1. / ( 2 * 3.14 ) ** 0.5 ) * vol * time ** 0.5 ) )
print( straddlePricer( 0.2, 1.0 ) == ( 2. * ( ( 1. / ( 2 * 3.14 ) ** 0.5 ) * vol * time ** 0.5 ) ) )

引数の名前を指定すれば、書く順番は問いません。

In [ ]:
print( straddlePricer( time=1.0, vol=0.2 ) )
print( straddlePricer( vol=0.2, time=1.0 ) )

便利になりました。次は、引数に初期値（デフォルト値）を持たせます。初期値を付けた引数は省略できます。まずは `time` だけを省略可能にします。

In [ ]:
def straddlePricer( vol, time=1.0 ):
    return 2. * ( ( 1 / ( 2 * 3.14 ) ** 0.5 ) * vol * time ** 0.5 )

In [ ]:
straddlePricer( 0.2 )

次は、`vol` と `time` の両方に初期値を付けます。

In [ ]:
def straddlePricer( vol=0.2, time=1.0 ):
    return 2. * ( ( 1 / ( 2 * 3.14 ) ** 0.5 ) * vol * time ** 0.5 )

こうすると、引数を渡さずに関数を呼べます。何も指定しなければ、`vol` は 0.2、`time` は 1.0 になります。

In [ ]:
straddlePricer()

In [ ]:
straddlePricer( 0.22 )

式の分母には π があります。ここまで使ってきた 3.14 は、おおまかな近似値です。もっと正確な値を使いたいところです。ライブラリ `numpy` に正確な値があります。まずインポートします。

In [ ]:
import numpy

`numpy` の関数は `numpy.xxxxx` と書いて使います。`xxxxx` の部分に、使いたい関数の名前を入れます。円周率は `numpy.pi` です。

In [ ]:
numpy.pi

`numpy` と毎回書くのは面倒です。短い別名を付けます。`numpy` の別名は `np` が慣例です。

In [ ]:
import numpy as np
import pandas as pd
import datetime as dt

In [ ]:
np.pi

`numpy` には、平方根を求める関数 `np.sqrt` もあります。

In [ ]:
np.sqrt( 4 )

`np.pi` と `np.sqrt` を使って、ストラドルの計算式を書き直します。少し正確になり、読みやすくもなります。

In [ ]:
def straddlePricer( vol=0.2, time=1.0 ):
    return 2. * ( ( 1 / np.sqrt( 2 * np.pi ) ) * vol * np.sqrt( time ) )

straddlePricer()

元の書き方と新しい書き方で、結果がどれだけ違うか見てみます。

In [ ]:
straddlePricer() - ( 2. * ( ( 1 / ( 2 * 3.14 ) ** 0.5 ) * vol * time ** 0.5 ) )

この例では、精度と読みやすさの差は大きくありません。それでも、場面によっては効いてきます。`numpy` には他にもいろいろな機能があります。たとえば乱数を作れます。

In [ ]:
np.random.rand()

使える関数を調べるには、`np.` のあとで Tab キーを押します。候補が表示されます。

In [ ]:
#np.

`dir` を使っても調べられます。`np` に何が含まれているか一覧できます。

In [ ]:
dir(np)

ストラドルはモンテカルロ法でも計算できます。資産価格の動きを再現するために、正規分布の乱数を使います。

In [ ]:
def straddlePricerMC(vol=0.2, time=1.0, mcPaths=100):
    dailyVol = vol / np.sqrt( 252. )
    resultSum = 0
    for p in range( mcPaths ):
        resultSum += np.abs( np.prod( ( 1 + np.random.normal( 0, dailyVol, int( round( time * 252 ) ) ) ) ) - 1 )
    return resultSum / mcPaths

straddlePricerMC()

新しい要素がたくさん出てきました。1行ずつ見ていきます。

分散は時間に比例します。日次のボラティリティは、次のどちらでも求められます。

1. 分散を時間で割り、平方根を取る
2. ボラティリティ（分散の平方根）を、時間の平方根で割る
    
ふだんはボラティリティで考えるので、2番目の方が分かりやすいです。どちらを使っても構いません。

In [ ]:
# 選択肢1の方法
np.sqrt( vol ** 2 / 252 )

In [ ]:
# 2つの方法を比べる
vol = 0.2
var = vol ** 2
sqrtVarOverTime = np.sqrt( var / 252 )
volOverSqrtTime = vol / np.sqrt( 252 )
valuesEqual = np.isclose( sqrtVarOverTime, volOverSqrtTime )
print( f'sqrtVarOverTime = {sqrtVarOverTime}\nvolOverSqrtTime = {volOverSqrtTime}\nAre they close? {valuesEqual}' )

次の行は地味です。合計を累積する変数 `resultSum` を 0 で初期化しています。これを忘れるとエラーになります。

In [ ]:
resultSum = 0

次はループです。ここでは `for` ループを使います。`range(mcPaths)` の要素を1つずつ取り出して繰り返します。では、`range(mcPaths)` とは何でしょうか。`range` は組み込み関数です。0 から x-1 までの整数を順に返します。

In [ ]:
range10 = range( 10 )
lst = list( range10 )
print( lst )
print( len( lst ) )

この例では、`p` の値そのものは使いません。こういうときは `_` と書くのが慣例です。ループを `mcPaths` 回回したいだけだからです。デフォルトでは100回回ります。

In [ ]:
def straddlePricerMC( vol=0.2, time=1.0, mcPaths=100 ):
    dailyVol = vol / np.sqrt( 252. )
    resultSum = 0
    for _ in range( mcPaths ):
        resultSum += np.abs( np.prod( 1 + ( np.random.normal( 0, dailyVol, int( round( time * 252 ) ) ) ) ) - 1 )
    return resultSum / mcPaths

straddlePricerMC()

ループの中で何をしているか、少しずつ見ていきます。一番内側の呼び出しから順に追います。まず、`np.random.normal` が何をする関数か調べます。ドキュメントの見方は2つあります。

1. help
2. ?

In [ ]:
help(np.random.normal)
# np.random.normal?

`help` を見ると、`np.random.normal` の引数が分かります。引数は、平均、標準偏差、作る乱数の個数（size）の3つです。省略すると平均 0、標準偏差 1 になり、乱数は1つだけ返ります。

In [ ]:
np.random.normal()

次に、平均 0、標準偏差に日次ボラティリティを指定して呼び出します。平均 0 は、ドリフトがないという意味です。満期までの日数分のリターンをまとめて作ります。

In [ ]:
time = 1
nDays = time * 252
dailyVol = vol / np.sqrt( 252. )
print( nDays )

np.random.normal( 0, dailyVol, nDays )

資産のリターンの時系列ができたら、ストラドルの価格が分かります。見たいのは満期時点の価格です。ストラドルは ATM で組まれているとします。価格は、初期値 1 からのずれの絶対値で表せます。

In [ ]:
np.random.seed( 42 ) # 乱数の種を固定する。2回とも同じ結果になる

returns = np.random.normal( 0, dailyVol, time * 252 )
priceAtMaturity = np.prod( 1 + returns )
changeAtMaturity = priceAtMaturity - 1
absChangeAtMaturity = np.abs( changeAtMaturity )
print( absChangeAtMaturity )

# 1行にまとめた書き方
np.random.seed( 42 )
print( np.abs( np.prod( 1 + ( np.random.normal( 0, dailyVol, time * 252 ) ) ) - 1 ) )

上の処理をもう少し詳しく見ます。ここでは pandas と perspective という2つのライブラリを使って、作業を楽にします。

In [ ]:
import pandas as pd
from perspective import psp

simulatedAsset = pd.DataFrame( np.random.normal( 0, dailyVol, time * 252 ) + 1, columns=['return'] )
simulatedAsset['price'] = ( 1 * simulatedAsset['return'] ).cumprod()
psp( simulatedAsset )

`for` ループは、上の処理を `mcPaths` 回繰り返しているだけです。最後に平均を取ると、ストラドルの期待値になります。

In [ ]:
mcPaths = 100
resultSum = 0.
for _ in range(mcPaths):
    resultSum += np.abs( np.prod( 1 + np.random.normal( 0., dailyVol, time * 252 ) ) - 1 )
print( resultSum / mcPaths )

この価格は、最初の計算式の結果にかなり近い値です。パス数を増やせば、もっと近づくはずです。

In [ ]:
straddlePricerMC(mcPaths=2000)

パス数 2000 でも、まだ元の価格に収束していません。パス数を増やすと、計算時間も増えます。Jupyter では、関数の実行時間を簡単に測れます。

In [ ]:
%timeit straddlePricerMC(mcPaths=2000)

かなり速いです。パス数をずっと増やせます。

In [ ]:
print(f"1 path: {straddlePricerMC(mcPaths=1)}")
print(f"2000 path: {straddlePricerMC(mcPaths=2000)}")
print(f"5000 path: {straddlePricerMC(mcPaths=5000)}")
print(f"10000 path: {straddlePricerMC(mcPaths=10000)}")
print(f"100000 path: {straddlePricerMC(mcPaths=100000)}")
print(f"Closed form approximation: {straddlePricer()}")

モンテカルロの実装は、もっと速くできます。乱数の系列を1回でまとめて作る方法です。`np.random.normal` の `size` 引数を思い出してください。

In [ ]:
nDays = time * 252
size = (nDays, 15)
simulatedAsset = pd.DataFrame(np.random.normal(0, dailyVol, size))
simulatedAsset = (1 + simulatedAsset).cumprod()

simulatedAsset.tail()

できました。matplotlib でグラフに描いてみます。

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.style.use('fivethirtyeight')

fig = plt.figure(figsize=(8,6))
ax = plt.axes()
_ = ax.plot(simulatedAsset)

この方法を、pandas を使ったモンテカルロ計算に組み込みます。

In [ ]:
def straddlePricerMCWithPD(vol=0.2, time=1, mcPaths=100000):
    dailyVol = vol / ( 252 ** 0.5 )
    randomPaths = pd.DataFrame( np.random.normal( 0, dailyVol, ( time*252, mcPaths ) ) )
    price = ( ( 1 + randomPaths ).prod() - 1 ).abs().sum() / mcPaths
    return price

straddlePricerMCWithPD()